In [2]:
""" import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0))
"""

' import torch\nprint(torch.cuda.is_available(), torch.cuda.get_device_name(0))\n'

# Create and run a local RAG pipeline

## Goals for RAG

* Retrieval - Find relevant info given a query, e.g. "what are the macronutrients and what do they do?" -> retrieves passages of text realted to the macronutrients from a nutrition textbok.
* Augmented - Take the relevant info and augment our input(prompt) to an LLM with that relevant info
* Generation - Take the first two steps and pass them to an LLM for generative outputs

## Main Goal
improve the outputs of LLMs by:
1. Preventing hallucinations
2. Work with custom data


## Use Cases:
* Textbook Q&A - find releveant passages to the questions you have.

In [3]:
import os
import requests

# Get PDF document path
pdf_path = "sustainability-action-plan.pdf"

# Download PDF
if not os.path.exists(pdf_path):
    print(f"[INFO] File doesn't exist, downloading...")

    #Enter the URL of the PDF
    url = "https://cityofwinterpark.org/docs/departments/natural-resources-sustainability/sustainability-division/sustainability-action-plan/sustainability-action-plan-annual-report-2023.pdf"

    #The local filename to save the downloaded file
    filename=pdf_path

    # Browser-like User-Agent required: the site returns 403 for python-requests' default
    headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
    )
}


    #Send a GET request to the URL
    response = requests.get(url, headers=headers, timeout=30)

    #Check if the request was successful
    if response.status_code ==200:
        # Open the file and save it
        with open(filename, "wb") as file:
            file.write(response.content)

        print(f"[INFO] The file has been downloaded and saved as {filename}")
    else:
        print(f"[INFO] Failed to download the file. Status code: {response.status_code}")

else:
    print(f"File {pdf_path} exists.")



File sustainability-action-plan.pdf exists.


In [4]:
%pip install tqdm
%pip install PyMuPDF

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [5]:
import pymupdf # requires: !pip install PyMuPDF
from tqdm.auto import tqdm

def text_formatter(text: str) -> str:
    """Performs minor formatting on text."""
    cleaned_text= text.replace("\n"," ").strip()

    return cleaned_text

PRINTED_PAGE_OFFSET = 2
def open_and_read_pdf(pdf_path: str) -> list[dict]:
    doc = pymupdf.open(pdf_path)
    pages_and_texts=[]
    
    for page_number, page in tqdm(enumerate(doc)):
        if page_number < PRINTED_PAGE_OFFSET:
            continue # skip the two pages before the title page
        
        text = page.get_text()
        text = text_formatter(text=text)
        pages_and_texts.append({"page_number": page_number - PRINTED_PAGE_OFFSET + 1,
                                "page_char_count": len(text),
                                "page_word_count": len(text.split(" ")),
                                "page_sentence_count": len(text.split(". ")),
                                "page_token_count": len(text) / 4, # 1 token = ~ 4 characters
                                "text": text})

    return pages_and_texts
pages_and_texts = open_and_read_pdf(pdf_path=pdf_path)
pages_and_texts[:2]

0it [00:00, ?it/s]

[{'page_number': 1,
  'page_char_count': 406,
  'page_word_count': 114,
  'page_sentence_count': 1,
  'page_token_count': 101.5,
  'text': 'City of Winter Park  Sustainability Action Plan                  January 24th, 2024                            Presented by:   Gloria Eby, Natural Resources and Sustainability Director     Sara Miller, Sustainability Manager   Mia Brady, Sustainability Specialist   Keep Winter Park Beautiful & Sustainable Advisory Board           Certified WCAG 2.1 Level AA compliant and usable with JAWS and NVDA by SWH'},
 {'page_number': 2,
  'page_char_count': 2873,
  'page_word_count': 155,
  'page_sentence_count': 22,
  'page_token_count': 718.25,
  'text': '2    Contents   Background ........................................................................................................................................... 3  Vision and Purpose ......................................................................................................................

In [6]:
import pandas as pd
pages_and_texts[9]

{'page_number': 10,
 'page_char_count': 1771,
 'page_word_count': 277,
 'page_sentence_count': 4,
 'page_token_count': 442.75,
 'text': '10  o Parks and Recreation summer camp program incorporates Orange County Public Schools (OCPS) Summer Food Program o Summer Feeding - Orange County Public Schools (ocps.net) • Winter Park Library o Master gardening basics:  5 classes, partnership with UF/IFAS Extension orange County o Seed library: 3 grow seasons; 9,000 seeds distributed per year • Rollins College o Food pantry for students – Students can either come to the pantry and take what they need or place an order online, which will be processed at the C-Store and stored in a pre-packed bag o Sam Justice is staff contact - sjustice@rollins.edu o Urban farm collaboration - providing fresh produce to the food pantry on a regular basis • SeniorsFirst o https://www.seniorsfirstinc.org/ o Provides meals-on-wheels to seniors in Winter Park • Winter Park High School o Willies Mart – “Willie’s Mart p

In [7]:

df = pd.DataFrame(pages_and_texts)
df.head()

,page_number,page_char_count,page_word_count,page_sentence_count,page_token_count,text
0,1,406,114,1,101.50,City of Winter Park Sustainability Action Pla...
1,2,2873,155,22,718.25,2 Contents Background ......................
2,3,2448,420,12,612.00,3 Background Vision and Purpose The 2022 Su...
3,4,2954,505,16,738.50,"4 In 2012, the city’s Environmental Review an..."
4,5,2551,411,7,637.75,5 indicators to benchmark city status in effo...


## Table Preprocessing

The goal is to extract tables from the PDF while preserving the relationships between column headers and row values

In [8]:
doc = pymupdf.open(pdf_path)

for page_number, page in enumerate(doc):
    tables = page.find_tables().tables

    if tables:
        print(f"Page {page_number}: {len(tables)} table(s) found")

Page 5: 1 table(s) found
Page 6: 1 table(s) found
Page 12: 1 table(s) found
Page 13: 1 table(s) found
Page 14: 1 table(s) found
Page 15: 1 table(s) found
Page 18: 2 table(s) found
Page 19: 1 table(s) found
Page 20: 1 table(s) found
Page 22: 2 table(s) found
Page 23: 1 table(s) found
Page 25: 2 table(s) found
Page 26: 1 table(s) found
Page 27: 1 table(s) found
Page 28: 1 table(s) found
Page 31: 1 table(s) found
Page 32: 1 table(s) found
Page 33: 1 table(s) found
Page 37: 2 table(s) found
Page 38: 1 table(s) found
Page 39: 1 table(s) found
Page 40: 1 table(s) found
Page 42: 2 table(s) found
Page 43: 1 table(s) found
Page 45: 2 table(s) found
Page 46: 1 table(s) found
Page 47: 1 table(s) found
Page 51: 1 table(s) found
Page 52: 2 table(s) found
Page 55: 2 table(s) found
Page 57: 1 table(s) found


In [9]:
# Automatically extract every table

all_tables = []

for page_number, page in enumerate(doc):
    tables = page.find_tables().tables

    for table_index, table in enumerate(tables):
        table_df = table.to_pandas()

        all_tables.append({
            "page_number" : page_number,
            "table_index": table_index,
            "data": table_df
        })

print(f"Extracted {len(all_tables)} tables")

Extracted 39 tables


# Inspect pages with Tables
## Goals:
1. Turn the tables into dataframe to inspect the actual rows and columns instead of treating the table as one long string

In [10]:
# Inspect the first table
display(all_tables[0]["data"])
# Check which page it came from
print(all_tables[0]["page_number"])

,2012-2013 KWPB&S Board Members,Col1,2014-2015 KWPB&S Board Members,Col3
0,"Mary Dipboye, Chair",James (Bob) Robinson,"Michael Poole, Chair",Mark Roush
1,"Stephen Pategas, V. Chair",Pat Schoknecht,"Stephen Pategas, V. Chair",Bruce Thomas
2,Michele Hipp,Julia Tensfeldt,Michele Hipp,Steven DiClemente
3,Michael Poole,Kent Tse,Raymond Randall,Mary Dipboye
4,Raymond Randall,Laura Walda,Pat Schoknecht,John Tapp
5,John Rife,Carol Kostick,Julia Tensfeldt,Fred Kosiewski
6,Lucy Roberts,Mark Roush,Laura Walda,Cathy Blanton
7,Joseph Robillard,,Carol Shenck (Kostick),


5


In [11]:
display(all_tables[3]["data"])
print(all_tables[3]["page_number"])

,Projected\nImplementation\nYear,Action,Accomplished,Status,Responsible\nDepartment/\nDivision
0,2022,Conduct Renewable Energy\nFeasibility Study,Yes,Completed,"Sustainability,\nQuanta, & WP\nElectric Utilit..."
1,2023,Establish community grants\nfor food security ...,No,None,Sustainability
2,2022,Implement an Artificial Turf\nOrdinance which ...,Yes,Yes,"Sustainability,\nPublic Works"
3,2022,Adopt Backyard Chicken\nProgram (exp. Septembe...,Yes,Yes,"Sustainability,\nPlanning"
4,2023,"Upon feasibility study\nresults, pass resoluti...",No,Initiated,"Sustainability,\nWP Electric\nUtility"
5,2023,Conduct a Climate Risk and\nVulnerability Asse...,No,Initiated,"Natural\nResources &\nSustainability,\nEmergen..."


13


In [12]:
display(all_tables[13]["data"])
print(all_tables[13]["page_number"])

,Col0,Indicator Description,2012\nBaseline,2021\nStatus,2022\nStatus,2025\nTarget,2035\nTarget
0,W-1,Water usage in residential buildings\n[gallons...,"127,777","130,849","127,606",TBD,TBD
1,W-2,Water usage in commercial buildings\n[gallons/...,"294,098","303,788","315,099",TBD,TBD
2,W-3,Reclaimed water usage [million\ngallons/year]2,144.5,91.61,87.97,10%\nincrease,30%\nincrease
3,W-4,Residential Rebates provided\nannually,-,34,19,75,150


25


In [13]:
display(all_tables[32]["data"])
print(all_tables[32]["page_number"])

,Col0,Climate Resiliency,Col2,Conduct Renewable Energy Feasibility Study
0,Climate Resiliency,None,None,Implement an Artificial Turf Ordinance which w...
1,Climate Resiliency,None,None,Adopt Backyard Chicken Program (exp. September...
2,Water,None,None,Create water conservation education to residen...
3,Water,None,None,Increase public awareness of Florida-friendly ...
4,None,Water,None,None
5,,Community Engagement and Green,,Re-participate in America In Bloom’s annual na...
6,None,Economy,None,None
7,,Local Government Operations,,Solicit proposals for energy conservation audi...
8,Local Government Operations,None,None,Conduct Renewable Energy Feasibility Study – 5...
9,Natural Resources,None,None,Provide education on pollution prevention of n...


47


In [14]:
display(all_tables[34]["data"])
print(all_tables[34]["page_number"])

,Col0,Battery Energy Storage Technology,Col2,Col3,2023,Col5,Col6,2035,Col8,Col9,2042,Col11,Col12,2050,Col14
0,Residential BESS–2.5 hr,Residential BESS–2.5 hr,None,"$1,443","$1,443",None,"$1,359","$1,359",None,"$1,359","$1,359",None,,"$1,359",
1,None,None,None,None,None,None,None,None,None,None,None,None,None,,None
2,Commercial BESS–4 hr,None,None,$774,None,None,$729,None,None,$729,None,None,$729,None,None
3,Utility Scale BESS–4 hr (ESS-4),None,None,$514,None,None,$484,None,None,$484,None,None,,$484,
4,None,None,None,None,None,None,None,None,None,None,None,None,None,,None


52


In [15]:
display(all_tables[35]["data"])
print(all_tables[35]["page_number"])

,Col0,Solar PV Technology,Col2,Col3,2023,Col5,Col6,2035,Col8,Col9,2042,Col11,Col12,2050,Col14
0,,Residential Rooftop Solar,,,"$3,400",,,"$2,892",,,"$2,892",,,"$2,892",
1,"Commercial Solar PV, Rooftop (Dsolar-CommRoof)",None,None,"$2,121",None,None,"$1,804",None,None,"$1,804",None,None,"$1,804",None,None
2,,"Commercial Solar PV, Ground-mount (Dsolar-Comm...",,,"$2,236",,,"$1,902",,,"$1,902",,,"$1,902",
3,Utility Scale Solar PV (USolar),None,None,"$1,331",None,None,"$1,253",None,None,"$1,253",None,None,"$1,253",None,None
4,Green Hydrogen-Fueled CT (CT-Hydrogen),None,None,"$1,561",None,None,"$1,828",None,None,"$1,828",None,None,"$1,828",None,None
5,Biofuel Internal Combustion Reciprocating Engi...,None,None,$520,None,None,$552,None,None,$552,None,None,$552,None,None


52


In [16]:
display(all_tables[36]["data"])
print(all_tables[36]["page_number"])

,Year,Col1,Col2,Scenario\n3A,Col4,Col5,Scenario\n4A,Col7,Col8,Scenario\n4B,Col10,Col11
0,20-year PVRR – 2023 to 2042 ($M),None,None,505,None,None,637,None,None,558,None,None
1,,LCOE based on 20-year PVRR ($/MWh),,,90,,,113,,,99,


55


In [17]:
display(all_tables[37]["data"])
print(all_tables[37]["page_number"])

,Year,Col1,Col2,Cumulative\nCity Rooftop\nPV (MW),Col4,Col5,Cumulative\nCity Ground\nMount PV\n(MW),Col7,Col8,Cumulative\nTotal MW),Col10,Col11,Cumulative\nCost ($M),Col13,Col14
0,2035,None,None,0.98,None,None,0.98,None,None,1.96,None,None,$3.8,None,None
1,,2042,,,0.98,,,2.95,,,3.94,,,$7.5,
2,2050,None,None,0.98,None,None,2.95,None,None,3.94,None,None,$7.5,None,None


55


In [18]:
display(all_tables[38]["data"])
print(all_tables[38]["page_number"])

,Year,Col1,Col2,Annualized\nCost\n($/MWh),Col4,Col5,Annualized\nCost\n($/MWh)\nescalated at\n3% after\n2042,Col7,Col8,LCOE\n($/MWh)\nwith a 3%\nescalation,Col10,Col11,Maximum\nAcceptable\nAnnualized\nPower Costs,Col13,Col14
0,,2023,,,$72,,,,,,$70,,,,
1,2024,None,None,$72,None,None,,None,None,$73,None,None,,None,None
2,,2025,,,$72,,,,,,$75,,,,
3,2026,None,None,$74,None,None,,None,None,$77,None,None,,None,None
4,,2027,,,$75,,,,,,$79,,,,
5,2028,None,None,$77,None,None,,None,None,$82,None,None,,None,None
6,,2029,,,$78,,,,,,$84,,,,
7,2030,None,None,$77,None,None,,None,None,$87,None,None,,None,None
8,,2031,,,$76,,,,,,$89,,,,
9,2032,None,None,$80,None,None,,None,None,$92,None,None,,None,None


57


# Check the structure of every extracted table and report possible problems

In [19]:
diagnostics = []

for i, item in enumerate(all_tables):
    table = item["data"].copy()

    # Treat blank strings and whitespace as missing
    table = table.replace(r"^\s*$", pd.NA, regex=True)

    # Identify columns with no values at all
    empty_columns = table.columns[table.isna().all()].tolist()

    # Count missing cells
    total_cells = table.size
    missing_cells = table.isna().sum().sum()

    missing_percent = (
        round(missing_cells / total_cells * 100, 1)
        if total_cells else 0
    )

    # How many columns have only one distinct nonempty value?
    constant_columns = []
    for col in table.columns:
        values = table[col].dropna().astype(str).str.strip()
        if len(values) > 0 and values.nunique() == 1:
            constant_columns.append(col)

    diagnostics.append({
        "table_index": i,
        "pdf_page_index": item["page_number"],
        "rows": table.shape[0],
        "columns": table.shape[1],
        "empty_columns": empty_columns,
        "missing_percent": missing_percent,
        "constant_columns": constant_columns,
        "duplicate_headers": table.columns.duplicated().any()
    })

diagnostics_df = pd.DataFrame(diagnostics)

display(
    diagnostics_df.sort_values(
        "missing_percent",
        ascending=False
    ).reset_index(drop=True)
)

,table_index,pdf_page_index,rows,columns,empty_columns,missing_percent,constant_columns,duplicate_headers
0,38,57,28,15,"[Col2, Col5, Col8, Col11, Col14]",77.1,[Col13],False
1,34,52,5,15,"[Col2, Col5, Col8, Col11, Col14]",74.7,"[Battery Energy Storage Technology, 2023, 2035...",False
2,37,55,3,15,"[Col2, Col5, Col8, Col11, Col14]",66.7,"[Col1, Cumulative\nCity Rooftop\nPV (MW), Col4...",False
3,36,55,2,12,"[Col2, Col5, Col8, Col11]",66.7,"[Year, Col1, Scenario\n3A, Col4, Scenario\n4A,...",False
4,35,52,6,15,"[Col2, Col5, Col8, Col11, Col14]",66.7,[],False
5,32,47,14,4,[Col2],55.4,[],False
6,6,18,5,7,[],22.9,[],False
7,2,12,5,11,"[Col8, Col10]",21.8,[],False
8,0,5,8,4,[],6.2,[],False
9,9,20,5,5,[],0.0,"[Accomplished, Status]",False


In [20]:
import random
random.sample(pages_and_texts, k=3)

[{'page_number': 42,
  'page_char_count': 2347,
  'page_word_count': 429,
  'page_sentence_count': 2,
  'page_token_count': 586.75,
  'text': '42  Projected  Implementation  Year  Action  Accomplished  Status  Responsible  Department/  Division  2023  Explore feasibility of an incentive program  for EV Charging Station installation in  multifamily and commercial properties   No  None  Sustainability,   Building & Permitting,  WP Electric Utility   2023  Develop EV infrastructure needs assessment  and master plan   Yes  Ongoing  Sustainability,   WP Electric Utility,  Planning & Zoning  2025  Explore opportunities to pilot electric  shuttle (Smart Bus)   No  Initiated  City Administration,  Economic   Development, Planning  & Zoning,  Sustainability  2025  Implement public and private rideshare  partnership opportunities that will promote  the use of the cities SunRail System    No  No  Economic   Development,  Planning & Zoning,   SunRail, Uber, & Lyft  Continue Annually Maintain Elect

In [21]:
df.describe().round(2)

,page_number,page_char_count,page_word_count,page_sentence_count,page_token_count
count,57.0,57.00,57.00,57.00,57.00
mean,29.0,2050.35,353.84,9.28,512.59
std,16.6,701.22,113.45,7.93,175.30
min,1.0,406.00,114.00,1.00,101.50
25%,15.0,1559.00,279.00,2.00,389.75
50%,29.0,1958.00,366.00,7.00,489.50
75%,43.0,2421.00,420.00,15.00,605.25
max,57.0,3876.00,639.00,33.00,969.00


In [22]:
# Drop Heading and Table of contents
records_df = records_df[~records_df.page_number.isin([1, 2])].reset_index(drop=True)

NameError: name 'records_df' is not defined

Interpreting the token count
Caution:

1.  LLMs don't deal with infinite tokens
2. Embedding models don't deal with infinite tokens